# Dataset v3: frozen LRW encoder + horizontal flip

Controlled validation-only experiment. The visual encoder remains frozen. The
only change relative to `uniform_shuffle` is adding horizontally flipped copies
of every training mouth clip. Validation stays unchanged and test is not loaded.


## 1. Подключить Google Drive и получить зависимости

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%pip -q install gdown

## 2. Скачать официальный LRW-код и найти dataset v3

In [ ]:
from pathlib import Path
import copy
import random
import shutil
import subprocess
import sys
import time
import zipfile

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
upstream_dir = Path('/content/Lipreading_using_Temporal_Convolutional_Networks')
upstream_store = drive_root / 'upstream_lrw_pretrained'
upstream_store.mkdir(parents=True, exist_ok=True)

if not upstream_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git',
        str(upstream_dir),
    ], check=True)

checkpoint_path = upstream_store / 'lrw_resnet18_mstcn_video.pth'
if not checkpoint_path.exists():
    downloaded = gdown.download(
        id='1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD',
        output=str(checkpoint_path),
        quiet=False,
    )
    if downloaded is None:
        raise RuntimeError('Official LRW checkpoint download failed.')

extract_dir = Path('/content/lipreading_data_v3')
relative_split_dir = (
    Path('03_splits')
    / 'ml_splits_mouth_crops_landmark_dataset_v3'
    / 'speaker_top10'
)
expected_split_dir = extract_dir / 'ru_dataset' / relative_split_dir
if not expected_split_dir.exists():
    matches = list(Path('/content/drive').rglob(
        'colab_lipreading_dataset_v3_train_val.zip'
    ))
    if not matches:
        raise FileNotFoundError(
            'Upload colab_lipreading_dataset_v3_train_val.zip to Google Drive.'
        )
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0], 'r') as archive:
        archive.extractall(extract_dir)
    print('Dataset restored from:', matches[0])

sys.path.insert(0, str(upstream_dir))
data_root = extract_dir / 'ru_dataset'
split_dir = data_root / relative_split_dir
output_dir = drive_root / 'lrw_frozen_hflip_dataset_v3'
output_dir.mkdir(parents=True, exist_ok=True)
print('dataset:', data_root)
print('checkpoint:', checkpoint_path)
print('output:', output_dir)

## 3. Прочитать только train и validation

Test намеренно не загружается: по нему нельзя подбирать архитектуру или learning rate.

In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('Enable a GPU runtime before running this experiment.')

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}

assert len(train_df) == 1901
assert len(val_df) == 105
assert set(train_df['speaker_id']).isdisjoint(set(val_df['speaker_id']))
assert all((data_root / path).is_file() for path in train_df['clip_path'])
assert all((data_root / path).is_file() for path in val_df['clip_path'])

print('device:', device)
print('train:', len(train_df), sorted(train_df['speaker_id'].unique()))
print('validation:', len(val_df), sorted(val_df['speaker_id'].unique()))
print('vocab:', vocab)
print('TEST SPLIT IS NOT LOADED.')

## 4. Датасет видеоклипов

Вход модели: 24 grayscale-кадра, `88x88`, с LRW-нормализацией. Здесь аугментаций нет, чтобы измерить только эффект partial fine-tuning.

In [ ]:
def load_lrw_video(path, num_frames=24, input_size=96, crop_size=88):
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frames = []
    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (input_size, input_size))
        offset = (input_size - crop_size) // 2
        frame = frame[offset:offset + crop_size, offset:offset + crop_size]
        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - 0.421) / 0.165)
    cap.release()

    if not frames:
        raise ValueError(f'No frames loaded: {path}')
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())
    return torch.from_numpy(np.stack(frames)).float().unsqueeze(0)


class EndToEndVideoDataset(Dataset):
    def __init__(self, metadata, data_root):
        self.metadata = metadata.reset_index(drop=True)
        self.data_root = data_root

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        video = load_lrw_video(self.data_root / row['clip_path'])
        label = word_to_idx[row['word']]
        return video, label


def make_video_loaders(seed, batch_size=8):
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        EndToEndVideoDataset(train_df, data_root),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        EndToEndVideoDataset(val_df, data_root),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    return train_loader, val_loader

preview_train_loader, _ = make_video_loaders(seed=42)
sample_x, sample_y = next(iter(preview_train_loader))
print('input:', sample_x.shape, sample_x.min().item(), sample_x.max().item())
print('labels:', sample_y.tolist())

## 5. LRW encoder + BiGRU

Сначала загружаются официальные LRW-веса. Затем frozen baseline с `uniform shuffle` и seed 42 точно воспроизводится на сохранённых LRW-признаках. Его лучшие веса BiGRU становятся стартом для partial fine-tuning.

In [ ]:
from lipreading.model import Lipreading


class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, num_classes),
        )

    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


class PartialFineTuneLipReader(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, videos):
        features = self.encoder(
            videos, lengths=[videos.shape[2]] * videos.shape[0]
        )
        return self.head(features)


tcn_options = {
    'num_layers': 4,
    'kernel_size': [3, 5, 7],
    'dropout': 0.2,
    'dwpw': False,
    'width_mult': 1,
}
encoder = Lipreading(
    modality='video',
    num_classes=500,
    tcn_options=tcn_options,
    backbone_type='resnet',
    relu_type='swish',
    width_mult=1.0,
    use_boundary=False,
    extract_feats=True,
)
official_checkpoint = load_torch_file(checkpoint_path)
encoder.load_state_dict(
    official_checkpoint.get('model_state_dict', official_checkpoint), strict=True
)
head = BiGRUClassifier(len(vocab))

model = PartialFineTuneLipReader(encoder, head).to(device)
assert hasattr(model.encoder.trunk, 'layer4')
print('Model is ready.')

## 6. Frozen feature cache and validation metrics


In [ ]:
class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in cache['words']], dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


def compute_metrics(labels, predictions):
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(
            labels, predictions, average='macro', zero_division=0
        ),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'predicted_classes': len(set(predictions)),
    }


def evaluate_head(head, loader, criterion):
    head.eval()
    loss_sum = 0.0
    labels = []
    predictions = []
    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = head(features)
            loss_sum += criterion(logits, targets).item() * targets.size(0)
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    metrics = compute_metrics(labels, predictions)
    metrics['loss'] = loss_sum / len(labels)
    return metrics


feature_root = drive_root / 'controlled_lrw_frontend_bigru_dataset_v3'
train_cache_path = feature_root / 'dataset_v3_train_lrw_resnet18_features.pt'
val_cache_path = feature_root / 'dataset_v3_val_lrw_resnet18_features.pt'
if not train_cache_path.exists() or not val_cache_path.exists():
    raise FileNotFoundError(
        'Frozen feature cache is missing. Run the dataset v3 LRW notebook first.'
    )
train_cache = load_torch_file(train_cache_path)
val_cache = load_torch_file(val_cache_path)
criterion = nn.CrossEntropyLoss()


## 7. Horizontal-flip augmentation and three-seed comparison

The original and flipped LRW feature sequences are concatenated only for
training. The unchanged validation speaker is evaluated after every epoch.


In [ ]:
class HorizontalFlipTrainDataset(EndToEndVideoDataset):
    def __getitem__(self, index):
        video, label = super().__getitem__(index)
        return torch.flip(video, dims=[3]), label

# Restore the official LRW encoder and freeze it completely.
model.encoder.load_state_dict(official_checkpoint.get('model_state_dict', official_checkpoint), strict=True)
for parameter in model.encoder.parameters():
    parameter.requires_grad = False
model.encoder.eval()

flip_loader = DataLoader(
    HorizontalFlipTrainDataset(train_df, data_root),
    batch_size=16, shuffle=False, num_workers=2, pin_memory=True
)
flip_features = []
flip_words = []
with torch.inference_mode():
    for step, (videos, targets) in enumerate(flip_loader, 1):
        videos = videos.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=True):
            features = model.encoder(videos, lengths=[videos.shape[2]] * videos.shape[0])
        flip_features.append(features.float().cpu())
        flip_words.extend(vocab[i] for i in targets.tolist())
        if step % 25 == 0 or step == len(flip_loader):
            print(f'feature extraction: {step}/{len(flip_loader)}', flush=True)

flip_features = torch.cat(flip_features)
assert flip_features.shape == train_cache['features'].shape
assert flip_words == train_df['word'].tolist()
augmented_cache = {
    'features': torch.cat([train_cache['features'].float(), flip_features], dim=0),
    'words': list(train_cache['words']) + flip_words,
}
print('augmented features:', augmented_cache['features'].shape)

SEEDS = [42, 123, 2026]
BASELINE_BY_SEED = {42: 0.545393, 123: 0.560324, 2026: 0.547428}
augmented_dataset = FeatureSequenceDataset(augmented_cache)
val_dataset = FeatureSequenceDataset(val_cache)
results = []
best_heads = {}

for seed in SEEDS:
    set_reproducible_seed(seed)
    head = BiGRUClassifier(len(vocab)).to(device)
    generator = torch.Generator().manual_seed(seed)
    train_aug_loader = DataLoader(augmented_dataset, batch_size=32, shuffle=True, generator=generator, num_workers=0)
    val_feature_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
    optimizer_head = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    best_f1, best_loss, best_epoch = -1.0, float('inf'), 0
    best_head_state = None
    without_improvement = 0
    started = time.perf_counter()
    for epoch in range(1, 31):
        head.train()
        for features, targets in train_aug_loader:
            features, targets = features.to(device), targets.to(device)
            optimizer_head.zero_grad(set_to_none=True)
            loss = criterion(head(features), targets)
            loss.backward()
            optimizer_head.step()
        metrics = evaluate_head(head, val_feature_loader, criterion)
        improved = metrics['macro_f1'] > best_f1 or (metrics['macro_f1'] == best_f1 and metrics['loss'] < best_loss)
        if improved:
            best_f1, best_loss, best_epoch = metrics['macro_f1'], metrics['loss'], epoch
            best_head_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
            without_improvement = 0
        else:
            without_improvement += 1
        if without_improvement >= 7:
            break
    elapsed_seed = time.perf_counter() - started
    delta = best_f1 - BASELINE_BY_SEED[seed]
    results.append({'seed': seed, 'best_epoch': best_epoch, 'validation_macro_f1': best_f1, 'baseline_macro_f1': BASELINE_BY_SEED[seed], 'paired_delta': delta, 'epochs_completed': epoch, 'training_seconds': elapsed_seed})
    best_heads[seed] = best_head_state
    print(f'hflip seed={seed}: F1={best_f1:.4f}, delta={delta:+.4f}, epoch={best_epoch}', flush=True)

hflip_results = pd.DataFrame(results)
hflip_summary = {
    'macro_f1_mean': float(hflip_results.validation_macro_f1.mean()),
    'macro_f1_std': float(hflip_results.validation_macro_f1.std(ddof=1)),
    'paired_delta_mean': float(hflip_results.paired_delta.mean()),
    'wins': int((hflip_results.paired_delta > 0).sum()),
}
print(hflip_results)
print('summary:', hflip_summary)
hflip_results.to_csv(output_dir / 'frozen_hflip_multiseed_runs.csv', index=False)
torch.save({'results': results, 'heads': best_heads, 'vocab': vocab, 'test_used': False}, output_dir / 'frozen_hflip_multiseed_heads.pt')
print('DECISION:', 'KEEP HFLIP' if hflip_summary['paired_delta_mean'] >= 0.02 and hflip_summary['wins'] >= 2 else 'DO NOT KEEP HFLIP')
print('TEST SPLIT WAS NOT USED.')

summary_df = pd.DataFrame([hflip_summary])
summary_df.to_csv(output_dir / 'frozen_hflip_multiseed_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(SEEDS))
ax.bar(x - 0.18, [BASELINE_BY_SEED[s] for s in SEEDS], 0.36, label='original')
ax.bar(x + 0.18, hflip_results['validation_macro_f1'], 0.36, label='original + hflip')
ax.set_xticks(x, [str(seed) for seed in SEEDS])
ax.set_xlabel('Seed')
ax.set_ylabel('Validation macro-F1')
ax.set_title('Frozen LRW encoder: horizontal-flip augmentation')
ax.legend()
plt.tight_layout()
plt.savefig(output_dir / 'frozen_hflip_multiseed_comparison.png', dpi=160)
plt.show()


## 8. Decision rule

Keep horizontal flip only when it improves the paired mean macro-F1 by at least
`0.02` and wins on at least two of three seeds. Do not open the test split in
this notebook.
